# Solar Energy Forecaster - Step 3: Build the API

**What is an API?** A way for other programs to talk to your model.

Someone sends weather data → your API returns a sunlight prediction.

We use FastAPI because it's simple and auto-generates documentation.

---
## 3.1 Create the API file

Run the cell below. It creates `src/api.py` — the file that runs your API server.

Read through the code. Every line has a comment explaining what it does.

In [ ]:
import os
os.makedirs("../src", exist_ok=True)

api_code = '''
from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import json
import numpy as np
import pandas as pd

# --- Load the trained model ---
model = joblib.load("models/solar_model.pkl")
with open("models/feature_columns.json") as f:
    feature_columns = json.load(f)

# --- Create the API ---
app = FastAPI(
    title="Solar Energy Forecaster",
    description="Predict sunlight from weather conditions"
)

# --- Define what inputs we expect ---
class WeatherInput(BaseModel):
    clouds: float        # Cloud cover in %  (0-100)
    temperature: float   # Temperature in °C
    humidity: float      # Humidity in %     (0-100)
    rain: float          # Rainfall in mm    (0+)
    hour: int            # Hour of day       (0-23)

# --- Health check endpoint ---
@app.get("/")
def home():
    return {"status": "running", "model": "Solar Forecaster"}

# --- Prediction endpoint ---
@app.post("/predict")
def predict(weather: WeatherInput):
    # Put the input into a table (DataFrame) with correct column order
    input_data = pd.DataFrame([{
        "clouds": weather.clouds,
        "temperature": weather.temperature,
        "humidity": weather.humidity,
        "rain": weather.rain,
        "hour": weather.hour,
    }])

    # Make prediction
    prediction = model.predict(input_data)[0]

    # Sunlight can't be negative
    prediction = max(0, float(prediction))

    return {
        "predicted_sunlight_wm2": round(prediction, 1),
        "input": weather.dict()
    }
'''

with open("../src/api.py", "w") as f:
    f.write(api_code.strip())

print("Created: src/api.py")
print("\nThis file defines two endpoints:")
print("  GET  /         → Health check (is the server running?)")
print("  POST /predict  → Send weather data, get sunlight prediction")

---
## 3.2 Run the API

Open a **terminal** (not this notebook), navigate to your project root folder, and run:

```
uvicorn src.api:app --reload --port 8000
```

You should see:
```
INFO:     Uvicorn running on http://127.0.0.1:8000
```

Then open your browser and go to: **http://localhost:8000/docs**

You'll see interactive documentation where you can test the API!

---
## 3.3 Test the API with Python

While the API is running in the terminal, run the cell below to send it a test request.

In [ ]:
import requests

# Test 1: Sunny afternoon
response = requests.post("http://localhost:8000/predict", json={
    "clouds": 10,          # Almost no clouds
    "temperature": 32,     # Hot day
    "humidity": 50,        # Moderate humidity
    "rain": 0,             # No rain
    "hour": 12             # Noon
})
print("Test 1 - Sunny afternoon:")
print(response.json())

print()

# Test 2: Rainy monsoon morning
response = requests.post("http://localhost:8000/predict", json={
    "clouds": 95,          # Very cloudy
    "temperature": 26,     # Cooler
    "humidity": 90,        # Very humid
    "rain": 8,             # Heavy rain
    "hour": 10             # Morning
})
print("Test 2 - Rainy monsoon morning:")
print(response.json())

print()

# Test 3: Nighttime
response = requests.post("http://localhost:8000/predict", json={
    "clouds": 30,
    "temperature": 25,
    "humidity": 70,
    "rain": 0,
    "hour": 22             # 10pm
})
print("Test 3 - Nighttime (should be ~0):")
print(response.json())

---
## 3.4 Create requirements.txt

This file tells other people (and Docker) what packages your project needs.

In [ ]:
requirements = """fastapi==0.104.1
uvicorn==0.24.0
pandas==2.1.4
numpy==1.26.2
scikit-learn==1.3.2
joblib==1.3.2
requests==2.31.0
matplotlib==3.8.2
"""

with open("../requirements.txt", "w") as f:
    f.write(requirements.strip())

print("Created: requirements.txt")

---
## 3.5 Create the Dockerfile

Docker packages your entire app into a "container" that runs the same everywhere.

In [ ]:
dockerfile = """FROM python:3.10-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY src/ ./src/
COPY models/ ./models/

EXPOSE 8000

CMD ["uvicorn", "src.api:app", "--host", "0.0.0.0", "--port", "8000"]
"""

with open("../Dockerfile", "w") as f:
    f.write(dockerfile.strip())

print("Created: Dockerfile")
print()
print("To build and run:")
print("  docker build -t solar-forecaster .")
print("  docker run -p 8000:8000 solar-forecaster")
print()
print("Then test: http://localhost:8000/docs")

---
## What you can say in an interview

**Q: Why FastAPI?**
> "It auto-generates interactive documentation at /docs, it validates inputs automatically using Pydantic, and it's the most popular Python framework for ML APIs."

**Q: Why Docker?**
> "Docker ensures the app runs the same on my laptop and on AWS. It packages the code, the model file, and all dependencies into one portable container. No 'it works on my machine' problems."

**Q: How does the API work?**
> "A user sends a POST request with 5 weather values. The API loads the saved model, formats the input as a DataFrame, calls model.predict(), and returns the predicted sunlight in W/m². It takes about 5 milliseconds."